# Open4D basics

Each example does one thing. Run the sections you need.

Install Open4D from the checkout with `python -m pip install -e '.[player,open3d]'`.
V-DMC and the Gaussian research methods need their own setup, described in the
[README](../README.md). Select the notebook kernel for your installed environment.


## Make or load a sequence

This creates a moving wave, so you do not need to download a dataset.


In [ ]:
import open4d
from open4d.demo import mesh_sequence

sequence = mesh_sequence(frames=30)
print(len(sequence), "frames")


For your own data, use `sequence = open4d.load("my_frames", fps=30)`. The folder holds one OBJ or PLY file per frame.


## Encode

After building V-DMC, set `OPEN4D_VDMC_ENCODER` and `OPEN4D_VDMC_DECODER` to its programs. The codec is always chosen explicitly. This writes the existing V-DMC artifact; it does not define the future general `.o4d` format.


In [ ]:
encoded = open4d.encode(sequence, "wave.v4d", codec="vdmc")


## Decode

The decoded sequence reads its data from the saved artifact.


In [ ]:
decoded = open4d.decode(encoded)
print(len(decoded), "decoded frames")


## Visualize

Drag to rotate, scroll to zoom, and press space to pause. Close the window to continue. You can also pass `sequence` here to view the original without encoding.


In [ ]:
open4d.visualize(decoded)


In [ ]:
decoded.close()


## Reconstruct from depth

This makes three small depth images of a surface one metre from a synthetic camera. Zero depth means missing data. The result is one mesh for each timestamp.


In [ ]:
import numpy as np

depth = np.full((3, 48, 64), 1000, dtype=np.uint16)
reconstructed = open4d.reconstruct(depth, intrinsics=(60, 60, 31.5, 23.5))
print(len(reconstructed[0].geometry.triangles), "triangles")


For real data, load your depth array instead. It must have shape `(frames, height, width)` in millimetres, or use `depth_scale=1` for metres. Replace the synthetic intrinsics with your camera calibration `(fx, fy, cx, cy)` in pixels. Add `color=rgb` for aligned RGB images, and `camera_poses=` if the camera moves.


In [ ]:
open4d.visualize(reconstructed)
reconstructed.close()


## Stream

Use two Python processes. Start the receiver first:

```python
from open4d import receive

with receive() as frames:
    for frame in frames:
        print(frame.frame_index, len(frame.geometry.positions), "vertices")
```

Then run the sender in the other process:

```python
from open4d import stream
from open4d.demo import mesh_sequence

stream(mesh_sequence(frames=30))
```

This sends the mesh frames at their recorded timing over TCP on this computer.
It does not compress them. Both sides use port 7000; pass `host=` and `port=`
for another address. Use a trusted network or SSH tunnel for remote transport.


## Gaussian splats

After setting up Vega's CUDA environment, encode two Gaussian PLY frames:

```python
frames = [open4d.load_gaussians("frame_0000.ply"), open4d.load_gaussians("frame_0001.ply")]
encoded = open4d.encode(frames, "capture.vega", codec="vega")
decoded = open4d.decode(encoded)
```

Decoded Vega frames keep their learned color model in `frame.appearance`.
The current adapter supports a single native group per run.

QUEEN and 3DGStream build splats from calibrated camera images. With the method's
runtime set up, reconstruct and render a QUEEN scene:

```python
run = open4d.reconstruct("my_scene", "queen_output", method="queen")
video = run.render()
```

`method="3dgstream"` selects 3DGStream, whose viewing still uses its native viewer.
See the README for runtime paths and input requirements. Meshes and splats use
different representations; the Qt viewer and TCP stream currently take meshes.


In [ ]:
sequence.close()
